# Q2 – TF-IDF Manual Computation + Verification

Compute by hand for ShopSense Clothing reviews, then verify with code.

In [ ]:
import re
import math
import numpy as np
import pandas as pd
from collections import Counter

REVIEWS_PATH = "shopsense_reviews-2.csv"
TARGET_WORD  = "fabric"


In [ ]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns and "category" in df.columns
        df["review_text"] = df["review_text"].fillna("").astype(str)
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {path}")

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text.lower())
    return text.split()

df = load_reviews(REVIEWS_PATH)
clothing_df = df[df["category"] == "Clothing"].reset_index(drop=True)
all_tokenized     = [clean_text(t) for t in df["review_text"]]
clothing_tokenized = [clean_text(t) for t in clothing_df["review_text"]]

print("Total corpus:", len(df))
print("Clothing docs:", len(clothing_df))
print("Doc_42 text:", clothing_df.loc[42, "review_text"][:200])


## (a) TF('fabric', Doc_42), IDF('fabric', 10K corpus), TF-IDF('fabric', Doc_42) – Step by step

In [ ]:
def compute_tf_word(word, tokens):
    n = len(tokens)
    if n == 0:
        return 0.0
    count = tokens.count(word)
    return count / n

def compute_idf_word(word, all_tokenized, smooth=True):
    N  = len(all_tokenized)
    df_ = sum(1 for tokens in all_tokenized if word in tokens)
    if smooth:
        return math.log((N + 1) / (df_ + 1)) + 1
    return math.log(N / (df_ + 1)) + 1

doc42_tokens = clothing_tokenized[42]

tf_fabric  = compute_tf_word(TARGET_WORD, doc42_tokens)
idf_fabric = compute_idf_word(TARGET_WORD, all_tokenized)
tfidf_fabric = tf_fabric * idf_fabric

print("=== Step-by-step ===")
print(f"Doc_42 tokens      : {doc42_tokens}")
print(f"Count('fabric')    : {doc42_tokens.count(TARGET_WORD)}")
print(f"Doc_42 length      : {len(doc42_tokens)}")
print(f"TF('fabric')       : {doc42_tokens.count(TARGET_WORD)} / {len(doc42_tokens)} = {tf_fabric:.6f}")
N  = len(all_tokenized)
df_count = sum(1 for tokens in all_tokenized if TARGET_WORD in tokens)
print(f"N (corpus size)    : {N}")
print(f"DF('fabric')       : {df_count}  (docs containing 'fabric')")
print(f"IDF('fabric')      : log(({N}+1) / ({df_count}+1)) + 1 = {idf_fabric:.6f}")
print(f"TF-IDF('fabric')   : {tf_fabric:.6f} * {idf_fabric:.6f} = {tfidf_fabric:.6f}")


## (b) IDF('the') vs IDF('embroidery')

In [ ]:
def idf_summary(word, all_tokenized):
    N   = len(all_tokenized)
    df_ = sum(1 for tokens in all_tokenized if word in tokens)
    idf = math.log((N + 1) / (df_ + 1)) + 1
    print(f"Word: '{word}'")
    print(f"  DF = {df_} / {N} docs")
    print(f"  IDF = log(({N}+1)/({df_}+1)) + 1 = {idf:.6f}")
    return idf

idf_the        = idf_summary("the", all_tokenized)
idf_embroidery = idf_summary("embroidery", all_tokenized)


## Explanation

IDF('the') approaches 0 because 'the' appears in almost every document in the corpus.
When the document frequency is very close to N, the fraction inside the log approaches 1, and log(1) = 0, so IDF drops to near 0 (plus the +1 smoothing offset, it stays just above 1 but very low).

IDF('embroidery') is high because the word appears in very few documents.
When document frequency is small, the fraction inside the log is large, making the IDF value significantly above 1, which means the word is considered more informative and discriminating across documents.


## (c) Rebuttal: 'Why not just use word frequency?'

Word frequency alone counts how often a term appears in a document, but it does not account for how common the term is across all documents.
A word like "the" or "product" appears frequently in almost every review, so high raw frequency just reflects a universal pattern, not something meaningful about a specific document.
TF-IDF corrects for this by multiplying the term frequency with an inverse document frequency that penalizes words appearing everywhere, so only words that are frequent in a specific document AND rare globally get high scores, which makes them genuinely informative.


## Bonus: BM25 weighting (k1=1.5, b=0.75)

In [ ]:
def compute_bm25_score(query_tokens, doc_tokens, idf_dict, all_tokenized, k1=1.5, b=0.75):
    avg_dl  = sum(len(t) for t in all_tokenized) / len(all_tokenized)
    doc_len = len(doc_tokens)
    tf_dict = Counter(doc_tokens)
    score   = 0.0
    for term in query_tokens:
        if term not in idf_dict:
            continue
        tf    = tf_dict.get(term, 0)
        idf   = idf_dict[term]
        numer = tf * (k1 + 1)
        denom = tf + k1 * (1 - b + b * (doc_len / avg_dl))
        score += idf * (numer / denom)
    return score

query_tokens  = ["fabric", "quality", "material"]
idf_cache = {w: compute_idf_word(w, all_tokenized) for w in set(query_tokens)}

doc42_bm25 = compute_bm25_score(query_tokens, doc42_tokens, idf_cache, all_tokenized)
doc42_tfidf_manual = sum(
    compute_tf_word(w, doc42_tokens) * idf_cache[w]
    for w in query_tokens
)
print(f"BM25 score  (Doc_42): {doc42_bm25:.6f}")
print(f"TF-IDF sum  (Doc_42): {doc42_tfidf_manual:.6f}")

top5_bm25 = []
for i, tokens in enumerate(clothing_tokenized[:200]):
    s = compute_bm25_score(query_tokens, tokens, idf_cache, all_tokenized)
    top5_bm25.append((i, s))

top5_bm25.sort(key=lambda x: -x[1])
print("\nTop 5 Clothing docs by BM25:")
for rank, (idx, s) in enumerate(top5_bm25[:5], 1):
    print(f"  {rank}. Doc {idx} | score={s:.4f} | {clothing_df.loc[idx, 'review_text'][:100]}")


## BM25 vs TF-IDF: How scores change

BM25 uses a saturation function on term frequency so that very frequent terms in a document have diminishing returns.
A document that says 'fabric' 10 times gets a much lower boost over 5 times compared to plain TF-IDF, which scales linearly.
BM25 also penalizes longer documents through the `b` parameter (document length normalization), so short documents with the query term are rewarded more.
Overall, BM25 tends to give better ranking quality than plain TF-IDF for search tasks because it avoids over-rewarding repetitive documents.
